# IN15: Architecture Review Techniques and Trade-off Documentation

**Programme:** Advanced Agentic AI -- Production Engineering
**Track:** India | Walmart Global Tech Academy
**Module:** 5 -- AI Economics, Optimization and Architecture Review

## Objectives

By the end of this notebook you will be able to:

- Understand what an Architecture Review Board (ARB) expects from a GenAI proposal
- Anticipate and prepare for the tough questions reviewers will ask
- Write a structured trade-off note that another engineer can read six months later
- Build a scored decision matrix for technology selection
- Produce an Architecture Decision Record (ADR) in the standard format
- Generate a complete ARB-ready summary document for the Walmart Retail Assistant

In [1]:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'], check=True)
# print('Packages ready.')

In [2]:
import os, json
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv(override=True)
print('Environment loaded.')

Environment loaded.


## Section 1: The Architecture Review Board (ARB)

### What is an ARB?

An Architecture Review Board is a formal governance process in which senior engineers,
architects, and business stakeholders evaluate a proposed technical design before it
is approved for production deployment.

**The ARB's four primary concerns for a GenAI system:**

| Concern | Questions the board will ask |
|---|---|
| Risk | What happens when this fails? How do you detect it? How do you recover? |
| Cost | What is the monthly spend? What is the worst-case scenario? Who pays? |
| Maintainability | Who owns this in 18 months? Can a new engineer understand it? |
| Defensibility | Why this approach and not X? What did you rule out and why? |

**What the ARB is NOT looking for:**
- The most technically impressive solution
- A solution that uses the newest framework
- A demo that works on a laptop

**What the ARB IS looking for:**
- Evidence that you have thought through failure modes
- A cost model with real numbers
- A clear rollback plan
- Documentation that will outlast the original team

**The six components every GenAI ARB presentation must cover (per India TOC):**

1. Architecture choice -- agent / workflow / traditional software
2. Agent strategy -- orchestration pattern, tool selection
3. Evaluation strategy -- metrics, golden set, go / no-go criteria
4. Cost model -- projected token usage, monthly spend, optimisation levers
5. Deployment model -- CI/CD, rollback, incident response
6. Risk mitigation plan -- security, hallucinations, drift

## Section 2: Tough Questions to Prepare For

### The 20 Questions an ARB Will Ask

The following questions represent the most common ARB challenges for GenAI systems.
For each question, prepare a one-paragraph written answer BEFORE the review.

**Architecture:**
1. Why an agent and not a deterministic workflow? What does the agent add?
2. If the LLM API is unavailable, what does the user experience?
3. What is your maximum context window size and what happens when you exceed it?

**Cost:**
4. What is your projected monthly spend at 100,000 calls/day?
5. What is your worst-case monthly spend (no caching, peak load)?
6. What happens to cost if the user base doubles in six months?
7. Who approved the budget and what is the governance threshold?

**Quality:**
8. How do you know the answers are accurate? What is your faithfulness score?
9. What is your golden dataset and how was it curated?
10. What is your go / no-go threshold for a model version upgrade?

**Security:**
11. What prevents a user from extracting your system prompt?
12. How do you handle PII in queries (e.g., order numbers, names)?
13. Have you red-teamed this system? What did you find?

**Deployment:**
14. What is your rollback procedure if v2 degrades quality?
15. How long does a rollback take? Who executes it?
16. What is your CI/CD pipeline for prompt changes?

**Operations:**
17. What does your on-call runbook say for a latency spike?
18. How do you detect silent drift (quality degradation without errors)?
19. What is your data retention policy for conversation logs?
20. Who reviews the Langfuse observability dashboard and how often?

## Section 3: The Scored Decision Matrix

### What is a Decision Matrix?

A decision matrix is a structured scoring table that evaluates competing
technology options against a fixed set of criteria with explicit weights.
It converts subjective 'I prefer X' opinions into defensible, documented decisions.

**The four-axis framework (from Module 1):**

| Axis | Weight | What it measures |
|---|---|---|
| Cost | 30% | Total cost of ownership: token spend + engineering effort + ops |
| Latency | 25% | P95 end-to-end response time at production scale |
| Quality | 30% | Faithfulness, accuracy, task success rate |
| Maintainability | 15% | Ease of debugging, updating, and handing over |

**Scoring scale:** 1 (poor) to 5 (excellent) per criterion.
Weighted score = sum of (score * weight) across all criteria.

**What to remember:**
- The matrix does not make the decision -- it documents it.
  The engineer makes the final call; the matrix shows the reasoning.
- Weights must be agreed before scoring, not after.
  Post-hoc weight adjustment to favour a preferred option is bias.
- Include at least one option you ruled out. The ARB will ask.

In [3]:
def decision_matrix(options: list, criteria: list, scores: dict) -> list:
    """
    options : list of option names
    criteria: list of (name, weight) tuples -- weights must sum to 1.0
    scores  : dict {option: {criterion: score 1-5}}
    """
    results = []
    for opt in options:
        weighted = sum(
            scores[opt].get(c, 1) * w
            for c, w in criteria
        )
        results.append({'option': opt, 'weighted_score': round(weighted, 3),
                        'scores': scores[opt]})
    return sorted(results, key=lambda x: -x['weighted_score'])

# Architecture choice decision matrix: Agent vs Workflow vs Traditional
options  = ['RAG + Agent (LangGraph)', 'RAG + Deterministic Workflow', 'Traditional Search API']
criteria = [
    ('cost',            0.30),
    ('latency',         0.25),
    ('quality',         0.30),
    ('maintainability', 0.15),
]
scores = {
    'RAG + Agent (LangGraph)': {
        'cost': 2, 'latency': 3, 'quality': 5, 'maintainability': 3,
    },
    'RAG + Deterministic Workflow': {
        'cost': 4, 'latency': 4, 'quality': 4, 'maintainability': 5,
    },
    'Traditional Search API': {
        'cost': 5, 'latency': 5, 'quality': 2, 'maintainability': 5,
    },
}

results = decision_matrix(options, criteria, scores)

print('Architecture Decision Matrix (Walmart Retail Assistant):')
print(f'{"Option":<35} {"Cost":<6} {"Latency":<9} {"Quality":<9} {"Maint":<7} {"Weighted"}')
print('-' * 75)
for r in results:
    s = r['scores']
    print(f'{r["option"]:<35} {s["cost"]:<6} {s["latency"]:<9} {s["quality"]:<9} {s["maintainability"]:<7} {r["weighted_score"]}')

winner = results[0]
print()
print(f'Recommended: {winner["option"]} (score: {winner["weighted_score"]})')
print('Rationale: Highest quality for complex multi-intent retail queries;')
print('           cost managed via model routing and caching (IN13).')
print('           Ruled out: Traditional Search API -- cannot handle multi-intent queries.')

Architecture Decision Matrix (Walmart Retail Assistant):
Option                              Cost   Latency   Quality   Maint   Weighted
---------------------------------------------------------------------------
RAG + Deterministic Workflow        4      4         4         5       4.15
Traditional Search API              5      5         2         5       4.1
RAG + Agent (LangGraph)             2      3         5         3       3.3

Recommended: RAG + Deterministic Workflow (score: 4.15)
Rationale: Highest quality for complex multi-intent retail queries;
           cost managed via model routing and caching (IN13).
           Ruled out: Traditional Search API -- cannot handle multi-intent queries.


## Section 4: Architecture Decision Records (ADR)

### What is an ADR?

An Architecture Decision Record is a short, structured document that captures
a significant architectural decision: the context that forced a choice,
the options considered, the decision made, and the consequences accepted.

**Why ADRs matter:**
- Six months after a decision, the people who made it may have moved teams.
  An ADR ensures the reasoning survives the original team.
- ADRs prevent re-litigating the same decisions repeatedly.
  When someone asks 'why LangGraph and not CrewAI?', the ADR has the answer.
- ADRs are the primary evidence the ARB uses to assess decision quality.

**ADR standard format (used at Walmart Global Tech):**

```
Title:    ADR-NNN: Short title of the decision
Date:     YYYY-MM-DD
Status:   Proposed | Accepted | Superseded | Deprecated
Deciders: Names or roles of decision-makers

Context:
  What is the situation? What problem must be solved?
  What constraints are we operating under?

Decision:
  What did we decide? Be specific.

Options Considered:
  Option A: ... (pros / cons)
  Option B: ... (pros / cons)
  Option C: ... (pros / cons)

Consequences:
  Positive: What does this decision make easier?
  Negative: What does this decision make harder or more expensive?
  Risks:    What could go wrong? How is it mitigated?

Review Date:  When should this decision be revisited?
```

**What to remember:**
- Keep ADRs short -- max 1 page. Long ADRs are not read.
- Write the context first, decision second. Context is what makes the decision defensible.
- Every negative consequence must have a mitigation or acceptance note.
  'No known downsides' is a red flag to any ARB reviewer.

In [4]:
def generate_adr(
    number: int,
    title: str,
    status: str,
    deciders: list,
    context: str,
    decision: str,
    options: list,
    consequences: dict,
    review_date: str,
) -> str:
    """Generate a formatted ADR document as a string."""
    lines = [
        f'ADR-{number:03d}: {title}',
        '=' * 65,
        f'Date     : {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
        f'Status   : {status}',
        f'Deciders : {", ".join(deciders)}',
        '',
        'CONTEXT',
        '-' * 65,
        context,
        '',
        'DECISION',
        '-' * 65,
        decision,
        '',
        'OPTIONS CONSIDERED',
        '-' * 65,
    ]
    for opt in options:
        lines.append(f'  {opt["name"]}')
        lines.append(f'    Pros: {opt["pros"]}')
        lines.append(f'    Cons: {opt["cons"]}')
        lines.append('')
    lines += [
        'CONSEQUENCES',
        '-' * 65,
        f'  Positive : {consequences["positive"]}',
        f'  Negative : {consequences["negative"]}',
        f'  Risks    : {consequences["risks"]}',
        f'  Mitigation: {consequences["mitigation"]}',
        '',
        f'REVIEW DATE: {review_date}',
    ]
    return '\n'.join(lines)

adr_001 = generate_adr(
    number=1,
    title='Use LangGraph for Walmart Retail Assistant Orchestration',
    status='Accepted',
    deciders=['ML Platform Team', 'Engineering Manager', 'Principal Architect'],
    context=(
        'The Walmart Retail Assistant must handle multi-intent queries that require '
        'sequential tool calls (price lookup -> inventory check -> policy retrieval). '
        'A simple prompt-completion loop cannot maintain state across steps. '
        'LangChain LCEL was considered but lacks native graph-based state management '
        'required for conditional branching in the supervisor pattern.'
    ),
    decision=(
        'Use LangGraph with a Supervisor + Worker pattern. '
        'The supervisor node routes queries to specialised worker nodes '
        '(price_agent, order_agent, policy_agent). '
        'State is managed via a TypedDict shared across all nodes.'
    ),
    options=[
        {'name': 'LangGraph (Chosen)',
         'pros': 'Native graph state, conditional routing, human-in-the-loop support, active OSS',
         'cons': 'Higher learning curve; adds framework dependency'},
        {'name': 'LangChain LCEL',
         'pros': 'Simpler for linear chains; familiar to team',
         'cons': 'No native graph state; branching requires custom code'},
        {'name': 'Raw Python (no framework)',
         'pros': 'Zero framework dependency; full control',
         'cons': 'Reimplements retry, state, and tool dispatch from scratch; high maintenance'},
    ],
    consequences={
        'positive': 'Stateful multi-step queries work natively; easy to add new worker nodes',
        'negative': 'Team must learn LangGraph API; upgrade risk when LangGraph releases breaking changes',
        'risks':    'LangGraph API instability between minor versions',
        'mitigation': 'Pin LangGraph version; integration test suite must run on every dependency update',
    },
    review_date='2027-01-01 (or when LangGraph v2.0 is released)',
)

print(adr_001)
Path('ADR-001_LangGraph_Orchestration.txt').write_text(adr_001)
print()
print('Saved: ADR-001_LangGraph_Orchestration.txt')

ADR-001: Use LangGraph for Walmart Retail Assistant Orchestration
Date     : 2026-07-23
Status   : Accepted
Deciders : ML Platform Team, Engineering Manager, Principal Architect

CONTEXT
-----------------------------------------------------------------
The Walmart Retail Assistant must handle multi-intent queries that require sequential tool calls (price lookup -> inventory check -> policy retrieval). A simple prompt-completion loop cannot maintain state across steps. LangChain LCEL was considered but lacks native graph-based state management required for conditional branching in the supervisor pattern.

DECISION
-----------------------------------------------------------------
Use LangGraph with a Supervisor + Worker pattern. The supervisor node routes queries to specialised worker nodes (price_agent, order_agent, policy_agent). State is managed via a TypedDict shared across all nodes.

OPTIONS CONSIDERED
-----------------------------------------------------------------
  LangGraph (C

## Section 5: Writing a Trade-off Note That Lasts

### What is a Trade-off Note?

A trade-off note is a lightweight, narrative version of an ADR.
It is written for a mixed audience -- engineers AND business stakeholders --
and focuses on the business impact of the technical decision.

**Trade-off note structure (one page maximum):**

```
Problem:  One sentence. What problem were you solving?
Decision: One sentence. What did you choose?
Why:      Two to three sentences. The core reasoning.
Trade-off: One sentence each on cost, latency, quality, maintainability.
Risk:     What can go wrong, and what is the mitigation?
Revisit:  Under what conditions would you reverse this decision?
```

**The one-sentence test:**
If you cannot explain your decision in one sentence, you do not understand it well enough.
Practice this before the ARB.

**Common mistakes in trade-off documentation:**
- Writing only about what was chosen, not what was rejected
- Listing benefits without acknowledging any downside
- Using jargon that a non-ML engineer cannot parse
- Failing to state who made the decision and when
- Not specifying a revisit date

In [5]:
def trade_off_note(
    decision_id: str,
    problem: str,
    decision: str,
    why: str,
    trade_offs: dict,
    risk: str,
    mitigation: str,
    revisit_trigger: str,
    owner: str,
) -> str:
    """Generate a one-page trade-off note."""
    lines = [
        f'TRADE-OFF NOTE: {decision_id}',
        '=' * 60,
        f'Owner  : {owner}',
        f'Date   : {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
        '',
        f'PROBLEM\n  {problem}',
        '',
        f'DECISION\n  {decision}',
        '',
        f'WHY\n  {why}',
        '',
        'TRADE-OFFS',
    ]
    for axis, note in trade_offs.items():
        lines.append(f'  {axis.capitalize():<16}: {note}')
    lines += [
        '',
        f'RISK\n  {risk}',
        '',
        f'MITIGATION\n  {mitigation}',
        '',
        f'REVISIT WHEN\n  {revisit_trigger}',
    ]
    return '\n'.join(lines)

tn = trade_off_note(
    decision_id='TON-002: Model Selection -- gpt-4o-mini as Primary',
    problem=(
        'The Walmart Retail Assistant must handle 100,000 daily queries at a cost '
        'sustainable within the $1,500/month team budget.'
    ),
    decision='Route 70% of queries to gpt-4o-mini; reserve gpt-4-turbo for complex multi-intent queries only.',
    why=(
        'gpt-4o-mini costs 33x less than gpt-4-turbo per input token. '
        'In-house evaluation (IN11 benchmark) shows gpt-4o-mini achieves normalised judge score '
        'of 0.83 vs 0.87 for gpt-4-turbo on the Walmart golden dataset -- a delta of 0.04, '
        'within the accepted regression threshold of 0.05.'
    ),
    trade_offs={
        'cost':            '$120/month vs $1,800/month -- 93% reduction at 100k calls/day',
        'latency':         '320ms faster P95 on gpt-4o-mini (lower token cost = faster generation)',
        'quality':         '-0.04 normalised judge score; within regression gate; no ARB objection',
        'maintainability': 'Model routing adds one decision layer; documented in ADR-003',
    },
    risk='Complex queries routed incorrectly to gpt-4o-mini produce incomplete answers.',
    mitigation=(
        'Router logic is keyword-based and logged. Weekly spot-check of routed queries. '
        'Fallback to gpt-4o if router confidence < 0.8.'
    ),
    revisit_trigger=(
        'If monthly faithfulness score drops below 0.82 for any 7-day rolling window, '
        'or if gpt-4o-mini pricing increases by more than 50%.'
    ),
    owner='ML Platform Team, Walmart India',
)

print(tn)
Path('TON-002_Model_Selection.txt').write_text(tn)
print()
print('Saved: TON-002_Model_Selection.txt')

TRADE-OFF NOTE: TON-002: Model Selection -- gpt-4o-mini as Primary
Owner  : ML Platform Team, Walmart India
Date   : 2026-07-23

PROBLEM
  The Walmart Retail Assistant must handle 100,000 daily queries at a cost sustainable within the $1,500/month team budget.

DECISION
  Route 70% of queries to gpt-4o-mini; reserve gpt-4-turbo for complex multi-intent queries only.

WHY
  gpt-4o-mini costs 33x less than gpt-4-turbo per input token. In-house evaluation (IN11 benchmark) shows gpt-4o-mini achieves normalised judge score of 0.83 vs 0.87 for gpt-4-turbo on the Walmart golden dataset -- a delta of 0.04, within the accepted regression threshold of 0.05.

TRADE-OFFS
  Cost            : $120/month vs $1,800/month -- 93% reduction at 100k calls/day
  Latency         : 320ms faster P95 on gpt-4o-mini (lower token cost = faster generation)
  Quality         : -0.04 normalised judge score; within regression gate; no ARB objection
  Maintainability : Model routing adds one decision layer; documente

## Section 6: ARB Readiness Checklist

Use this checklist to verify you are fully prepared for the Architecture Review Board.
Every item must be checked before submitting your ARB deck.

In [6]:
ARB_CHECKLIST = {
    'Architecture': [
        'Decision matrix completed for architecture choice (agent / workflow / traditional)',
        'At least one option explicitly ruled out with documented reasoning',
        'ADR written and reviewed for each major technical decision',
    ],
    'Cost Model': [
        'Monthly spend projected at expected and peak load',
        'Worst-case scenario (no caching, all premium models) calculated',
        'Cost per call computed and within governance threshold ($0.05 flag)',
        'Budget approved and governance thresholds configured',
        'Chargeback model defined and team budget owner identified',
    ],
    'Evaluation': [
        'Golden dataset defined (minimum 20 records, 5 categories)',
        'All 10 evaluation metrics from Module 4 have pass/fail thresholds',
        'Evaluation scorecard generated and all metrics PASS',
        'Regression test baseline established (IN11)',
        'Go / No-Go criteria documented for future model upgrades',
    ],
    'Security': [
        'Prompt injection defence implemented and tested',
        'PII handling policy defined (no PII in logs)',
        'Red-team results documented with fixes applied',
        'Toxicity threshold configured (BLOCK > 0.30)',
    ],
    'Deployment': [
        'CI/CD pipeline for prompt and model changes defined',
        'Rollback procedure documented (target: < 30 minutes)',
        'On-call runbook written and reviewed',
        'Langfuse observability dashboard live in staging',
        'SLO defined (P95 < 3000ms) and alerting configured',
    ],
    'Risk': [
        'Hallucination mitigation documented (faithfulness threshold 0.85)',
        'Drift detection configured (weekly benchmark run)',
        'Circuit breaker implemented for external API failures',
        'Fallback responses defined for all failure modes',
    ],
}

def arb_readiness_check(completed: dict) -> dict:
    """Score ARB readiness. completed: {category: [completed item strings]}"""
    total_items   = sum(len(v) for v in ARB_CHECKLIST.values())
    passed_items  = 0
    category_scores = {}
    for cat, items in ARB_CHECKLIST.items():
        done = completed.get(cat, [])
        cat_passed = sum(1 for item in items if any(d.lower() in item.lower() for d in done))
        category_scores[cat] = {'passed': cat_passed, 'total': len(items)}
        passed_items += cat_passed
    return {
        'total_items':    total_items,
        'passed_items':   passed_items,
        'readiness_pct':  round(passed_items / total_items * 100, 1),
        'arb_ready':      passed_items == total_items,
        'by_category':    category_scores,
    }

# Print checklist
print('ARB READINESS CHECKLIST -- Walmart Retail Assistant')
print('=' * 60)
total = 0
for cat, items in ARB_CHECKLIST.items():
    print(f'\n{cat}:')
    for item in items:
        print(f'  [ ] {item}')
        total += 1
print(f'\nTotal items: {total}')
print('Complete all items before submitting ARB deck.')

ARB READINESS CHECKLIST -- Walmart Retail Assistant

Architecture:
  [ ] Decision matrix completed for architecture choice (agent / workflow / traditional)
  [ ] At least one option explicitly ruled out with documented reasoning
  [ ] ADR written and reviewed for each major technical decision

Cost Model:
  [ ] Monthly spend projected at expected and peak load
  [ ] Worst-case scenario (no caching, all premium models) calculated
  [ ] Cost per call computed and within governance threshold ($0.05 flag)
  [ ] Budget approved and governance thresholds configured
  [ ] Chargeback model defined and team budget owner identified

Evaluation:
  [ ] Golden dataset defined (minimum 20 records, 5 categories)
  [ ] All 10 evaluation metrics from Module 4 have pass/fail thresholds
  [ ] Evaluation scorecard generated and all metrics PASS
  [ ] Regression test baseline established (IN11)
  [ ] Go / No-Go criteria documented for future model upgrades

Security:
  [ ] Prompt injection defence implemen

## Summary

| Concept | Key Rule |
|---|---|
| ARB concerns | Risk, Cost, Maintainability, Defensibility -- in that order |
| Decision matrix | Agree weights before scoring; include at least one rejected option |
| ADR | Context first, decision second; every negative has a mitigation |
| Trade-off note | One page maximum; write for a mixed technical/business audience |
| ARB readiness | All 6 sections must be complete: arch / cost / eval / security / deployment / risk |

**Next: IN16 (Capstone Problem Statement) and IN17 (Capstone Solution)**
-- Apply everything from Modules 1-5 to design, implement, evaluate, and defend
a production Walmart Retail Assistant from scratch.